In [1]:
################################################################################
# File name:    "warren_town_boundary_matches.ipynb"
#
# Project title:    Boston Affordable Housing project (visting scholar porject)
#
# Description:    This is a version of the original boundary matching file used
#                 to match address points to zoning boundaries. It runs much
#                 faster than the original version and instead of matching 
#                 properties to zoning boundaries, they are matched to town 
#                 boundaries. The program exports a CSV and the output should
#                 only be used in the 20_warren_town_boundary_matches.do file
#
# Inputs:    ./zone_assignments_export.csv
#            ./town_boundary_analysis_final.shp
#
# Outputs:    ./warren_town_boundary_matches.csv
#
# Created:    05/23/2024
# Updated:    05/23/2024
#
# Author:    Nicholas Chiumenti
################################################################################

In [2]:
import os
import re
import shutil
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points

In [3]:
start_time = datetime.now()
print("Running closest_boundary_matches program...")

Running closest_boundary_matches program...


In [4]:
## load boundary data
# set boundary shape file path
boundary_path = "/home/nicholas/Python-Projects/town_boundary_analysis/town_boundary_analysis_final.shp"

# read in shape file as a geo dataframe
boundary_gdf = gpd.read_file(boundary_path)

## Prep the town boundary file
# rename variables b/c ArcGIS sucks and renamed everything badly
boundary_gdf = boundary_gdf.rename(columns = {"reg_left_f": "reg_left_fid",
                               "reg_right_": "reg_right_fid",
                               "muni_left_": "muni_left_fid",
                               "muni_right": "muni_right_fid",
                               "left_muni_": "left_muni_id",
                               "left_muni1": "left_muni_name",
                               "right_muni": "right_muni_id",
                               "right_mu_1": "right_muni_name",
                               "left_zo_us": "left_zo_usety",
                               "left_minlo": "left_minlotsize",
                               "left_mxht_": "left_mxht_eff",
                               "left_maxdu": "left_maxdu",
                               "left_dupac": "left_dupac_eff",
                               "left_mulfa": "left_mulfam",
                               "left_reg_t": "left_reg_type",
                               "left_LRID": "left_LRID",
                               "right_zo_u": "right_zo_usety",
                               "right_minl": "right_minlotsize",
                               "right_mxht": "right_mxht_eff",
                               "right_maxd": "right_maxdu",
                               "right_dupa": "right_dupac_eff",
                               "right_mulf": "right_mulfam",
                               "right_reg_": "right_reg_type",
                               "right_LRID": "right_LRID"})

# create a static unique id, NOTE THIS WILL BECOME THE NEW LAM_SEG VARIABLE
boundary_gdf["boundary_unique_id"] = boundary_gdf.index

# set all town names to upper case to match better
boundary_gdf["left_muni_name"] = boundary_gdf["left_muni_name"].str.upper() 
boundary_gdf["right_muni_name"] = boundary_gdf["right_muni_name"].str.upper() 

# fix naming for the borough towns to match better
boundary_gdf["left_muni_name"].replace({'MARLBOROUGH':'MARLBORO'}, inplace=True)
boundary_gdf["left_muni_name"].replace({'FOXBOROUGH':'FOXBORO'}, inplace=True)
boundary_gdf["left_muni_name"].replace({'SOUTHBOROUGH':'SOUTHBORO'}, inplace=True)
boundary_gdf["left_muni_name"].replace({'BOXBOROUGH':'BOXBORO'}, inplace=True)

boundary_gdf["right_muni_name"].replace({'MARLBOROUGH':'MARLBORO'}, inplace=True)
boundary_gdf["right_muni_name"].replace({'FOXBOROUGH':'FOXBORO'}, inplace=True)
boundary_gdf["right_muni_name"].replace({'SOUTHBOROUGH':'SOUTHBORO'}, inplace=True)
boundary_gdf["right_muni_name"].replace({'BOXBOROUGH':'BOXBORO'}, inplace=True)

ERROR 1: PROJ: proj_create_from_database: Open of /home/nicholas/.conda/envs/boston_zoning/share/proj failed
/tmp/ipykernel_5731/4153000153.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  boundary_gdf["left_muni_name"].replace({'MARLBOROUGH':'MARLBORO'}, inplace=True)
/tmp/ipykernel_5731/4153000153.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the in

In [5]:
## load properties data

data_path = "/home/nicholas/Python-Projects/town_boundary_analysis/warren_sample.csv" # note this is a sample dataset for testing, don't use it
#data_path = "/home/a1nfc04/Documents/boston_zoning_sdrive/python_programs/zone_assignments/zone_assignments_export.csv"

# import zone assignments w/ all columns as string
data_df = pd.read_csv(data_path, dtype=str)

# trim variables
data_df = data_df[["prop_id", 
                   "cousub_name", 
                   "warren_latitude", 
                   "warren_longitude"]]

# convert dataframe to geodataframe
data_gdf = gpd.GeoDataFrame(data_df, 
                            geometry = gpd.points_from_xy(data_df["warren_longitude"], data_df["warren_latitude"]),
                            crs = "EPSG:4269")

data_gdf = data_gdf.sample(n=1000)

In [6]:
# confirm the CRSs are the same
assert data_gdf.crs == boundary_gdf.crs, "crs between data_gdf and boundary_gdf do not match"

In [7]:
# match properties to the 'left' side of the boundaries
left_side_matches = data_gdf.merge(boundary_gdf, 
                                   how="inner",
                                   left_on=["cousub_name"],
                                   right_on=["left_muni_name"],
                                   indicator=True,
                                   suffixes=("_x", "_fid")
                                  )

left_side_matches.rename(columns={"_merge" : "lside_merge"}, inplace=True)

# tag as a left side match
left_side_matches.loc[:, "boundary_side"] = "LEFT"

# match properties to the 'right' side of the boundaries
right_side_matches = data_gdf.merge(boundary_gdf, 
                                   how="inner",
                                   left_on=["cousub_name"],
                                   right_on=["right_muni_name"],
                                   indicator=True,
                                   suffixes=("_x", "_fid")
                                  )

right_side_matches.rename(columns={"_merge" : "rside_merge"}, inplace=True)

# tag as a right side match
right_side_matches.loc[:, "boundary_side"] = "RIGHT"

In [8]:
# append the two matched dfs together into one
matches_df = pd.concat([left_side_matches, right_side_matches], ignore_index=True)

# drop any cases where the latitude or longitude were missing
matches_df = matches_df.dropna(subset=['warren_latitude', 'warren_longitude'])

In [9]:
## Calculate distance to nearest boundary and store nearest point

n=0

for i, row in matches_df.iterrows():
    
    n+=1
    
    # get the address and boundary geographies
    address_point = row["geometry_x"] # the address, point object
    boundary_line = row["geometry_fid"] # the boundary, line onject

    # return a line from address to the nearest point on boundary
    nearest_x, nearest_y = nearest_points(address_point, boundary_line)

    matches_df.loc[i, "nearest_point_dist"] = nearest_x.distance(nearest_y)    # distance to nearest point on boundary_line
    matches_df.loc[i, "nearest_point_lat"] = nearest_y.y                     # latitude coordinate of nearest point
    matches_df.loc[i, "nearest_point_lon"] = nearest_y.x                     # longitude coordinate of nearest point
    
    print(f"{n:,} of {len(matches_df):,} distances calculated", end="\r")


In [10]:
# this just creates a 'saved game' dataframe so you don't have to re-run stuff if something gets messed up
# save_point = matches_df.copy()

In [11]:
# sort by prop_id and distance (and unique boundary_id if there is a tie)
matches_df.sort_values(by=["prop_id", "nearest_point_dist", "boundary_unique_id"], ascending=True, inplace=True)

# keep the 1st closest match
matches_df = matches_df.groupby("prop_id").head(1)

# number the matches in order of distance
matches_df.loc[:, "match_num"] = matches_df.groupby("prop_id")["nearest_point_dist"].rank(method="first", ascending=True)

# save dataset as final
matches_df.sort_index(inplace=True)


In [12]:
# export dataframe to a csv file
matches_df.to_csv("warren_town_boundary_matches.csv")

In [13]:
matches_df.columns

Index(['prop_id', 'cousub_name', 'warren_latitude', 'warren_longitude',
       'geometry_x', 'reg_left_fid', 'reg_right_fid', 'muni_left_fid',
       'muni_right_fid', 'left_muni_id', 'left_muni_name', 'right_muni_id',
       'right_muni_name', 'left_zo_usety', 'left_minlotsize', 'left_mxht_eff',
       'left_maxdu', 'left_dupac_eff', 'left_mulfam', 'left_reg_type',
       'left_LRID', 'right_zo_usety', 'right_minlotsize', 'right_mxht_eff',
       'right_maxdu', 'right_dupac_eff', 'right_mulfam', 'right_reg_type',
       'right_LRID', 'Shape_Leng', 'geometry_fid', 'boundary_unique_id',
       'lside_merge', 'boundary_side', 'rside_merge', 'nearest_point_dist',
       'nearest_point_lat', 'nearest_point_lon', 'match_num'],
      dtype='object')